In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.metrics import r2_score
import os 
import glob
import math
import scipy.signal
import pwlf
import warnings


In [6]:
class Test:
    def __init__(self, df):
        self.df = df.copy()  # Avoid modifying the original DataFrame
        self.id = df['participant_ID'].iloc[0]
        self.date = df['visit_date'].iloc[0]
        self.name = f"{self.id}_{self.date}"
        
        # Assign 'Stage' only for 'EXERCISE' phase, then fill NaN values with 0
        self.df['Stage'] = np.nan
        exercise_mask = self.df['Phase'] == 'EXERCISE'
        self.df.loc[exercise_mask, 'Stage'] = self.df[exercise_mask].groupby(['Speed', 'Grade']).ngroup() + 1
        self.df['Stage']= self.df['Stage'].fillna(0)  # Ensure non-exercise rows have Stage 0
        
        # Convert 't' to timedelta and set it as index
        self.df['t'] = pd.to_timedelta(self.df['t'])
        self.df.set_index('t', inplace=True)
        
        self.lactate_df = self.prepare_lactate_df()
    
    def prepare_lactate_df(self):
        # Reset index to retrieve 't' column
        lactate_df = self.df.dropna(subset=['La-']).reset_index()
        lactate_df = lactate_df[['t', 'La-', 'Phase', 'Speed', 'Grade', 'Stage']]
        
        # Remove extreme lactate values (greater than 20)
        lactate_df = lactate_df[lactate_df['La-'] <= 20]
        
        # Apply logarithmic transformation, handling non-positive values
        lactate_df['log_lactate'] = np.log(lactate_df['La-'].replace(0, np.nan))
        lactate_df['log_speed'] = np.log(lactate_df['Speed'].replace(0, np.nan))
        
        return lactate_df

    def lt_ref_vals(self):
        lt1ref_index = (self.lactate_df['La-'].diff() >= 0.5)
        lt2ref_index = (self.lactate_df['La-'].diff() >= 1)
        LT1ref = self.lactate_df.loc[lt1ref_index.idxmax(), 'La-'] if lt1ref_index.any() else None
        LT2ref = self.lactate_df.loc[lt2ref_index.idxmax(), 'La-'] if lt2ref_index.any() else None
        LT1ref_speed = self.lactate_df.loc[lt1ref_index.idxmax(), 'Speed'] if lt1ref_index.any() else None
        LT2ref_speed = self.lactate_df.loc[lt2ref_index.idxmax(), 'Speed'] if lt2ref_index.any() else None
        results = {"LT1_ref":LT1ref, "LT2_ref":LT2ref, "LT1ref_speed":LT1ref_speed, "LT2ref_speed":LT2ref_speed}
        return results

    def lt_abs_vals(self):
        lt1abs_index = (self.lactate_df['La-'] >= 2)
        lt2abs_index = (self.lactate_df['La-'] >= 4)
        
        LT1abs = self.lactate_df.loc[lt1abs_index.idxmax(), 'La-'] if lt1abs_index.any() else None
        LT2abs = self.lactate_df.loc[lt2abs_index.idxmax(), 'La-'] if lt2abs_index.any() else None
        LT1abs_speed = self.lactate_df.loc[lt1abs_index.idxmax(), 'Speed'] if lt1abs_index.any() else None
        LT2abs_speed = self.lactate_df.loc[lt2abs_index.idxmax(), 'Speed'] if lt2abs_index.any() else None
        
        results = {
            "LT1_abs": LT1abs, 
            "LT2_abs": LT2abs, 
            "LT1abs_speed": LT1abs_speed, 
            "LT2abs_speed": LT2abs_speed
        }
        return results

    def v_slope_method(self, x_list, y_list):
        """Finds the optimal intersection point using the V-Slope method."""
        split_index = len(x_list) // 2
        best_index = split_index
        min_error = float('inf')
        
        for i in range(10, len(x_list) - 10):  # Ensure enough points for both regressions
            slope1, intercept1, _, _, _ = linregress(x_list[:i], y_list[:i])
            slope2, intercept2, _, _, _ = linregress(x_list[i:], y_list[i:])
            
            if slope1 < 1 and slope2 >= 1:  # Condition for V-slope method
                error = np.sum((y_list[:i] - (slope1 * x_list[:i] + intercept1))**2) + \
                        np.sum((y_list[i:] - (slope2 * x_list[i:] + intercept2))**2)
                if error < min_error:
                    min_error = error
                    best_index = i
        
        slope1, intercept1, _, _, _ = linregress(x_list[:best_index], y_list[:best_index])
        slope2, intercept2, _, _, _ = linregress(x_list[best_index:], y_list[best_index:])
        
        x_threshold = (intercept2 - intercept1) / (slope1 - slope2)
        y_threshold = slope1 * x_threshold + intercept1
        
        return best_index, slope1, intercept1, slope2, intercept2, x_threshold, y_threshold  
            
    def lt_log_semilog(self, plot=False):
        
    # === Prepare Data ===
        filtered_df = self.lactate_df.drop_duplicates(subset='Speed', keep='first')
        filtered_df = filtered_df[(filtered_df['Speed'] > 0) & (filtered_df['La-'] > 0)]

        log_speed = np.log(filtered_df['Speed'].values)
        log_lactate = np.log(filtered_df['La-'].values)
        speed = filtered_df['Speed'].values

        # === LT1: log(speed) vs log(lactate) ===
        my_pwlf1 = pwlf.PiecewiseLinFit(log_speed, log_lactate)
        breaks1 = my_pwlf1.fit(2)  # 2 segments
        x_intersect_lt1 = breaks1[1]
        y_intersect_lt1 = my_pwlf1.predict([x_intersect_lt1])[0]
        lt1_log = math.exp(y_intersect_lt1)
        lt1_log_speed = math.exp(x_intersect_lt1)
        lt1_log_r2 = my_pwlf1.r_squared()

        # === LT2: speed vs log(lactate) ===
        my_pwlf2 = pwlf.PiecewiseLinFit(speed, log_lactate)
        breaks2 = my_pwlf2.fit(2)
        x_intersect_lt2 = breaks2[1]
        y_intersect_lt2 = my_pwlf2.predict([x_intersect_lt2])[0]
        lt2_semilog = x_intersect_lt2
        lt2_semilog_speed = math.exp(y_intersect_lt2)
        lt2_semi_log_r2 = my_pwlf2.r_squared()

        # === Plotting ===
        if plot:
            fig, axs = plt.subplots(1, 2, figsize=(12, 5))

            # LT1 plot
            axs[0].scatter(log_speed, log_lactate, color='gray', label='Data')
            x_hat1 = np.linspace(min(log_speed), max(log_speed), 100)
            y_hat1 = my_pwlf1.predict(x_hat1)
            axs[0].plot(x_hat1, y_hat1, color='blue', label='pwlf fit')
            axs[0].scatter(x_intersect_lt1, y_intersect_lt1, color='black', zorder=3, label='LT1')
            axs[0].set_title('LT1: Log Speed vs Log Lactate')
            axs[0].set_xlabel('Log Speed')
            axs[0].set_ylabel('Log Lactate')
            axs[0].legend()
            axs[0].grid()

            # LT2 plot
            axs[1].scatter(speed, log_lactate, color='gray', label='Data')
            x_hat2 = np.linspace(min(speed), max(speed), 100)
            y_hat2 = my_pwlf2.predict(x_hat2)
            axs[1].plot(x_hat2, y_hat2, color='red', label='pwlf fit')
            axs[1].scatter(x_intersect_lt2, y_intersect_lt2, color='black', zorder=3, label='LT2')
            axs[1].set_title('LT2: Speed vs Log Lactate')
            axs[1].set_xlabel('Speed')
            axs[1].set_ylabel('Log Lactate')
            axs[1].legend()
            axs[1].grid()

            plt.tight_layout()
            plt.show()

        return {
            "LT1_log":  lt1_log,
            "LT1_log_speed": lt1_log_speed,
            "LT1_log_r2": lt1_log_r2,
            "LT2_semilog": lt2_semilog, 
            "LT2_semilog_speed": lt2_semilog_speed,
            "LT2_semi_log_r2": lt2_semi_log_r2
        }

    def vt1_vt2_vo2_vco2(self, plot=False):
        vo2 = self.df['VO2']
        vco2 = self.df['VCO2']
        ve = self.df['VE']
        
        # Define an initial split index to separate the two regression regions
        split_index = len(vo2) // 2
        
        # Function to find the optimal intersection point
        best_index = split_index
        min_error = float('inf')
        
        for i in range(10, len(vo2) - 10):  # Ensure enough points for both regressions
            slope1, intercept1, _, _, _ = linregress(vo2[:i], vco2[:i])
            slope2, intercept2, _, _, _ = linregress(vo2[i:], vco2[i:])
            
            if slope1 < 1 and slope2 >= 1:  # Condition for V-slope method
                error = np.sum((vco2[:i] - (slope1 * vo2[:i] + intercept1))**2) + \
                        np.sum((vco2[i:] - (slope2 * vo2[i:] + intercept2))**2)
                if error < min_error:
                    min_error = error
                    best_index = i
        
        # Compute best fit lines
        slope1, intercept1, _, _, _ = linregress(vo2[:best_index], vco2[:best_index])
        slope2, intercept2, _, _, _ = linregress(vo2[best_index:], vco2[best_index:])
        
        # Intersection point
        vt1_vo2_threshold = (intercept2 - intercept1) / (slope1 - slope2)
        vt1_vco2_threshold = slope1 * vt1_vo2_threshold + intercept1

        # Fit piecewise model with 2 segments (1 breakpoint)
        my_pwlf = pwlf.PiecewiseLinFit(vco2, ve)
        breaks = my_pwlf.fit(2)  # This returns x-values of breakpoints

        # VT2 is the breakpoint between segment 1 and 2
        vt2_vco2 = breaks[1]
        vt2_ve = my_pwlf.predict([vt2_vco2])[0]
        # Create plot
        if plot:
            fig, axs = plt.subplots(1, 2, figsize=(14, 6))
            # --- VT1 Plot ---
            axs[0].scatter(vo2, vco2, label='Data', color='lightgray')
            axs[0].plot(vo2[:best_index], slope1 * vo2[:best_index] + intercept1, 'b', label='Slope < 1')
            axs[0].plot(vo2[best_index:], slope2 * vo2[best_index:] + intercept2, 'r', label='Slope ≥ 1')
            axs[0].scatter(vt1_vo2_threshold, vt1_vco2_threshold, color='black', zorder=3, label='VT1')
            axs[0].set_xlabel('VO₂ (L/min)')
            axs[0].set_ylabel('VCO₂ (L/min)')
            axs[0].set_title('V-Slope Method (VT1)')
            axs[0].legend()
            axs[0].grid()

               # --- VT2 Plot using PWLF ---
            axs[1].scatter(vco2, ve, label='Data', color='lightgray')

            # Plot the fitted segments
            x_hat = np.linspace(min(vco2), max(vco2), 100)
            y_hat = my_pwlf.predict(x_hat)
            axs[1].plot(x_hat, y_hat, 'b-', label='PWLF Fit')

            # Mark VT2
            axs[1].scatter(vt2_vco2, vt2_ve, color='black', marker='^', zorder=3, label='VT2 (PWLF)')
            axs[1].axvline(vt2_vco2, color='black', linestyle='--', alpha=0.5)

            axs[1].set_xlabel('VCO₂ (L/min)')
            axs[1].set_ylabel('VE (L/min)')
            axs[1].set_title('Ventilatory Compensation Point (VT2)')
            axs[1].legend()
            axs[1].grid()
                
        return {"vt1_vo2_level_vo2_vo2":vt1_vo2_threshold, "vt1_vo2_level_vo2_vco2":vt1_vco2_threshold,
                'vt2_vco2': vt2_vco2, 'vt2_ve': vt2_vco2}
    
    def vt1_vt2_VE(self, window_size, plot=False):
        VE_VO2 = self.df['VE/VO2'].rolling(f'{window_size}s', min_periods=1).mean()
        VE_CO2 = self.df['VE/VCO2'].rolling(f'{window_size}s', min_periods=1).mean()

        # Convert time index to total seconds
        time = self.df.index.total_seconds()
        # Identify nadir: Minimum VE/VO2 value
        nadir_index = VE_VO2.idxmin()
        if pd.isna(nadir_index) or nadir_index not in self.df.index:
            raise ValueError("nadir_index is NaN or not in DataFrame index")

        nadir_time = self.df.index[self.df.index.get_loc(nadir_index)]  # Safer method to get position
        nadir_value = VE_VO2[nadir_index]

        # Find the first rise after nadir while VE/CO2 is constant or increasing
        rise_index = None

        nadir_seconds = nadir_time.total_seconds()  # Convert Timedelta to seconds
        end_seconds = self.df.index[-1].total_seconds()
        for index in VE_CO2.index[1:]:
            prev_index = VE_CO2.index[VE_CO2.index < index].max()
            if VE_CO2.loc[index] >= VE_CO2.loc[prev_index] and VE_VO2.loc[index] > nadir_value:
                rise_index = prev_index
                break

        if rise_index is not None:
            rise_time = rise_index.total_seconds()
            rise_VE_VO2 = VE_VO2[rise_index]
            rise_VE_CO2 = VE_CO2[rise_index]
            #print(f"First rise after nadir found at time {rise_time} sec with VE/VO2 = {rise_VE_VO2} and VE/CO2 = {rise_VE_CO2}")
        else:
            print("No first rise after nadir found.")
        # Find the deflection point of VE/CO2 after nadir_index
        deflection_index = None
        for index in VE_CO2.index[VE_CO2.index > nadir_index]:
            if (index.total_seconds() - nadir_seconds) < 100:
                continue  # Ensure at least 5 seconds have passed
            prev_index = VE_CO2.index[VE_CO2.index < index].max()
            next_index = VE_CO2.index[VE_CO2.index > index].min()
            
            if prev_index is not None and next_index is not None:
                prev_slope = VE_CO2.loc[index] - VE_CO2.loc[prev_index]
                next_slope = VE_CO2.loc[next_index] - VE_CO2.loc[index]
                
                if prev_slope > 0 and next_slope < 0:  # Detect peak or deflection
                    deflection_index = index
                    break

        if deflection_index is not None:
            deflection_time = deflection_index.total_seconds()
            deflection_VE_CO2 = VE_CO2[deflection_index]
            #print(f"Deflection point of VE/CO2 found at time {deflection_time} sec with VE/CO2 = {deflection_VE_CO2}")
        else:
            print("No deflection point of VE/CO2 found after nadir.")
        vt1_speed, vt1_grade = self.get_speed_grade_from_time(nadir_time)
        vt2_speed, vt2_grade = self.get_speed_grade_from_time(deflection_index) if deflection_index is not None else (None, None)

        if plot:
            fig, ax = plt.subplots(figsize=(8, 6))

            ax.plot(time, VE_VO2, color='b', label='VE/VO2', zorder=2)
            ax.plot(time, VE_CO2, color='r', label='VE/CO2', zorder=1)
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Value')
            ax.tick_params(axis='y')

            # Mark the nadir, first rise, and deflection points
            ax.scatter(nadir_seconds, nadir_value, color='black', zorder=3, label='Nadir')
            if rise_index is not None:
                ax.scatter(rise_time, rise_VE_VO2, color='green', zorder=3, label='First Rise')
            if deflection_index is not None:
                ax.scatter(deflection_time, deflection_VE_CO2, color='purple', zorder=3, label='Deflection Point')

            fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
            ax.grid(True)
            plt.title('VE/VO2 and VE/CO2 vs. Time')
            #plt.show()

        return {'vt1_time_VE': nadir_time, 'vt1_speed_VE': vt1_speed, 'vt1_grade_VE': vt1_grade,
                'vt2_time_VE': deflection_index if deflection_index is not None else None, 'vt2_speed_VE': vt2_speed, 'vt2_grade_VE': vt2_grade}

    def get_speed_grade_from_time(self, time):
        #time = pd.to_timedelta(time)

        # Slice everything up to and including 'time'
        sub_df = self.df.loc[:time]
        if sub_df.empty:
            raise ValueError(f"No data available at or before time {time}")

        row = sub_df.iloc[-1]  # Last row before or equal to time
        return row['Speed'], row['Grade']
     
    def vt1_vt2_pet(self, window_size, plot=False):
            # Apply rolling mean
        PetO2 = self.df['PetO2'].rolling(f'{window_size}s', min_periods=1).mean()
        PetCO2 = self.df['PetCO2'].rolling(f'{window_size}s', min_periods=1).mean()
        
        # Convert time index to total seconds
        time = self.df.index.total_seconds()
        # Identify nadir: Minimum PetO2 value
        nadir_index = PetO2.idxmin()
        if pd.isna(nadir_index) or nadir_index not in self.df.index:
            raise ValueError("nadir_index is NaN or not in DataFrame index")

        nadir_time = self.df.index[self.df.index.get_loc(nadir_index)]  # Safer method to get position
        nadir_value = PetO2[nadir_index]

        # Find the first rise after nadir while PetCO2 is constant or increasing
        rise_index = None

        nadir_seconds = nadir_time.total_seconds()  # Convert Timedelta to seconds
        end_seconds = self.df.index[-1].total_seconds()
        for index in PetCO2.index[1:]:
            prev_index = PetCO2.index[PetCO2.index < index].max()
            if PetCO2.loc[index] >= PetCO2.loc[prev_index] and PetO2.loc[index] > nadir_value:
                rise_index = prev_index
                break

        if rise_index is not None:
            rise_time = rise_index.total_seconds()
            rise_PetO2 = PetO2[rise_index]
            rise_PetCO2 = PetCO2[rise_index]
            #print(f"First rise after nadir found at time {rise_time} sec with PetO2 = {rise_PetO2} and PetCO2 = {rise_PetCO2}")
        else:
            print("No first rise after nadir found.")
        # Compute the first derivative (rate of change) --> after aerobic threshold
        mask = PetCO2.index > nadir_index
        PetCO2_post_nadir = PetCO2[mask]
        time_post_nadir = time[mask]

        # Derivatives
        dPetCO2_dt = np.gradient(PetCO2_post_nadir, time_post_nadir)
        d2PetCO2_dt2 = np.gradient(dPetCO2_dt, time_post_nadir)

        # Deflection: point of max acceleration after nadir
        deflection_idx_local = np.argmax(d2PetCO2_dt2)
        deflection_index = PetCO2_post_nadir.index[deflection_idx_local]
        deflection_time = deflection_index
        deflection_value = PetCO2_post_nadir.iloc[deflection_idx_local]
        # Print results
        #print(f"Deflection point found at time {deflection_time} sec with PetCO2 = {deflection_value}")
        # Plot PetO2 and PetCO2
        vt1_speed, vt1_grade = self.get_speed_grade_from_time(nadir_index)
        vt2_speed, vt2_grade = self.get_speed_grade_from_time(deflection_time) if deflection_index is not None else (None, None)
        if plot:
            fig, ax1 = plt.subplots(figsize=(8, 6))

            ax1.plot(time, PetO2, color='b', label='PetO2', zorder=2)
            ax1.set_xlabel('Time (s)')
            ax1.set_ylabel('PetO2 (L/min)', color='b')
            ax1.tick_params(axis='y', labelcolor='b')

            ax2 = ax1.twinx()
            ax2.plot(time, PetCO2, color='r', label='PetCO2', zorder=1)
            ax2.set_ylabel('PetCO2 (L/min)', color='r')
            ax2.tick_params(axis='y', labelcolor='r')

            # Mark the nadir, first rise, and deflection points
            ax1.scatter(nadir_index.total_seconds(), nadir_value, color='black', zorder=3, label='Nadir')
            if rise_index is not None:
                ax1.scatter(rise_time, rise_PetO2, color='green', zorder=3, label='First Rise')
            ax2.scatter(deflection_time.total_seconds(), deflection_value, color='purple', zorder=3, label='Deflection Point')

            fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
            ax1.grid(True)
            plt.title('PetO2 and PetCO2 vs. Time')
            #plt.show()

        return {'vt1_time_pet':nadir_time, 'vt1_speed_pet':vt1_speed, 'vt1_grade_pet':vt1_grade,
                'vt2_time_pet':deflection_time, 'vt2_speed_pet':vt2_speed, 'vt2_grade_pet':vt2_grade}
    
    def get_VO2_peak_time_metrics(self, window_size=30, rq_threshold=1.0):
        """
        Calculates the peak VO2 value over a rolling window (default 30 seconds), and collects associated metrics.

        Returns:
            dict: Contains the peak VO2 value, start/end times, and corresponding physiological metrics.
        """
        # Compute rolling average (requires t to be the datetime index)
        rolling_avg = self.df['VO2'].rolling(f'{window_size}s').mean()

        max_avg = rolling_avg.max()
        end_time = rolling_avg.idxmax()
        start_time = end_time - pd.Timedelta(seconds=30)

        # Handle edge case if there's no peak found
        if pd.isna(max_avg) or pd.isna(start_time) or pd.isna(end_time):
            return {"VO2_peak": None, "start_time": None, "end_time": None}

        # Check if Stage is NaN at end_time
        try:
            if pd.isna(self.df.loc[end_time, 'Stage']):
                grade = self.df.loc[start_time, 'Grade']
                speed = self.df.loc[start_time, 'Speed']
                stage = self.df.loc[start_time, 'Stage']
            else:
                grade = self.df.loc[end_time, 'Grade']
                speed = self.df.loc[end_time, 'Speed']
                stage = self.df.loc[end_time, 'Stage']
        except KeyError:
            # fallback in case the exact time index isn't available (rare)
            nearest_end_idx = self.df.index.get_indexer([end_time], method='nearest')[0]
            end_time = self.df.index[nearest_end_idx]
            start_time = end_time - pd.Timedelta(seconds=30)
            grade = self.df.loc[end_time, 'Grade']
            speed = self.df.loc[end_time, 'Speed']
            stage = self.df.loc[end_time, 'Stage']

        # Slice data between start and end time
        peak_window_df = self.df.loc[start_time:end_time]

        rq_peak = peak_window_df['RQ'].mean()
        hr_peak = peak_window_df['HR'].mean()
        eem_peak = peak_window_df['EEm'].mean()
        fat_pct_peak = peak_window_df['Fat'].mean()
        cho_pct_peak = peak_window_df['CHO'].mean()
        vo2_kg_peak = peak_window_df['VO2/Kg'].mean()

        # First value after VO2 peak
        post_peak = self.df.loc[self.df.index >= end_time]
        pre_peak = self.df.loc[self.df.index < end_time]
        # Lactate: First valid value after peak, or last valid value before if none after
        if not post_peak['La-'].dropna().empty:
            lactate_post = post_peak['La-'].dropna().iloc[0]
        else:
            lactate_post = pre_peak['La-'].dropna().iloc[-1] if not pre_peak['La-'].dropna().empty else None

        # First non-NaN RPE after VO2 peak
        rpe_post = post_peak['Dyspnea'].dropna().iloc[0] if not post_peak['Dyspnea'].dropna().empty else None

        # Boolean for whether RQ exceeds the threshold
        true_vo2max = rq_peak > rq_threshold if not pd.isna(rq_peak) else False

        return {
            "VO2_peak": max_avg,
            "start_time": start_time,
            "end_time": end_time,
            "GradePeak": grade,
            "SpeedPeak": speed,
            "MarkerPeak": stage,
            "RQPeak": rq_peak,
            "HRPeak": hr_peak,
            "EEMPeak": eem_peak,
            "Fat%Peak": fat_pct_peak,
            "CHO%Peak": cho_pct_peak,
            "VO2/kgPeak": vo2_kg_peak,
            "Lactate-VO2Peak": lactate_post,
            "RPE-VO2peak": rpe_post,
            "True VO2max": true_vo2max
        }
    
    def get_all_metrics(self, vo2_window_size=30, vt_window_size=15, rq_thresh=1.0):
        funcs_with_args = {
            'get_VO2_peak_and_time_metrics': {'window_size': vo2_window_size, 'rq_threshold': rq_thresh},
            'lt_abs_vals': {},
            'lt_ref_vals': {},
            'lt_log_semilog': {},
            'vt1_vt2_vo2_vco2': {},
            'vt1_vt2_VE': {'window_size': vt_window_size},  
            'vt1_vt2_pet': {'window_size': vt_window_size},
        }

        results = {}
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning)
            for func_name, kwargs in funcs_with_args.items():
                try:
                    method = getattr(self, func_name)
                    result = method(**kwargs)
                    if isinstance(result, dict):
                        results.update(result)
                    else:
                        results[func_name] = result
                except Exception as e:
                    results[f"{func_name}_error"] = str(e)
        return results

    def is_steady_state(self, O2_series, threshold=0.02):
        variance = O2_series.std() / O2_series.mean()
        return variance < threshold

    def get_stage_metrics(self, stage_df, summary_df, window_size_seconds, rq_threshold=1.0):
        if stage_df.empty:
            print(f"Warning: Empty DataFrame for stage {stage_df.name}")
            return

        # If the index is in Timedelta format, temporarily convert it to datetime for filtering
        if isinstance(stage_df.index, pd.TimedeltaIndex):
            base_time = pd.Timestamp('1970-01-01')
            datetime_index = base_time + stage_df.index

            # Define the end and start time for the window in datetime
            end_time_dt = datetime_index[-1]
            start_time_dt = end_time_dt - pd.Timedelta(seconds=window_size_seconds)

            # Filter using datetime
            window_df = stage_df[(datetime_index >= start_time_dt)]

            # Convert the start and end times back to timedelta
            start_time = start_time_dt - base_time
            end_time = end_time_dt - base_time

        elif not isinstance(stage_df.index, pd.DatetimeIndex):
            # If index is not datetime nor timedelta, try to convert
            stage_df = stage_df.set_index(pd.to_datetime(stage_df.index))
            end_time = stage_df.index[-1]
            start_time = end_time - pd.Timedelta(seconds=window_size_seconds)
            window_df = stage_df[stage_df.index >= start_time]

        else:
            # Already a datetime index
            end_time = stage_df.index[-1]
            start_time = end_time - pd.Timedelta(seconds=window_size_seconds)
            window_df = stage_df[stage_df.index >= start_time]

        if window_df.empty:
            return  # Not enough data

        # Create new row in summary_df
        summary_df.loc[stage_df.name] = np.nan
        summary_df.loc[stage_df.name, 'Start Time'] = str(start_time).split('.')[0]
        summary_df.loc[stage_df.name, 'End Time'] = str(end_time).split('.')[0]

        summary_df.loc[stage_df.name, 'Velocity (km/h)'] = stage_df['Speed'].iloc[0]
        summary_df.loc[stage_df.name, 'Grade (%)'] = stage_df['Grade'].iloc[0]
        summary_df.loc[stage_df.name, 'Stage (#)'] = stage_df['Stage'].iloc[0]

        last_rpe = stage_df['Dyspnea'].iloc[-1]
        if isinstance(last_rpe, str):
            summary_df.loc[stage_df.name, 'RPE (#)'] = int(last_rpe.split("_")[-1])
        else:
            summary_df.loc[stage_df.name, 'RPE (#)'] = last_rpe

        last_lactate = stage_df['La-'].dropna().iloc[-1] if not stage_df['La-'].dropna().empty else np.nan
        summary_df.loc[stage_df.name, 'Lactate (mmol/L)'] = last_lactate

        summary_df.loc[stage_df.name, 'VO2 (mL/min)'] = window_df['VO2'].mean()
        summary_df.loc[stage_df.name, 'VO2 (mL/min/kg)'] = window_df['VO2/Kg'].mean()
        summary_df.loc[stage_df.name, 'HR (bpm)'] = window_df['HR'].mean()
        summary_df.loc[stage_df.name, 'RQ'] = window_df['RQ'].mean()

        summary_df.loc[stage_df.name, 'True VO2max? T/F'] = summary_df.loc[stage_df.name, 'RQ'] > rq_threshold
        summary_df.loc[stage_df.name, 'Reach Steady State? T/F'] = self.is_steady_state(window_df['VO2'])
        summary_df.loc[stage_df.name, 'O2 variance'] = window_df['VO2'].var()
        
    def create_test_summary(self, window_size_seconds=50, rq_threshold=1.0):
        summary_df = pd.DataFrame({
            'Start Time': pd.Series(dtype='str'),
            'End Time': pd.Series(dtype='str'),
            'Velocity (km/h)': pd.Series(dtype='float'),
            'Grade (%)': pd.Series(dtype='float'),
            'Stage (#)': pd.Series(dtype='float'),
            'RPE (#)': pd.Series(dtype='float'),
            'Lactate (mmol/L)': pd.Series(dtype='float'),
            'VO2 (mL/min)': pd.Series(dtype='float'),
            'VO2 (mL/min/kg)': pd.Series(dtype='float'),
            'HR (bpm)': pd.Series(dtype='float'),
            'RQ': pd.Series(dtype='float'),
            'True VO2max? T/F': pd.Series(dtype='bool'),
            'Reach Steady State? T/F': pd.Series(dtype='bool'),
            'O2 variance': pd.Series(dtype='float')
        })

        for stage_name, group in self.df.groupby('Stage'):
            group.name = stage_name
            self.get_stage_metrics(group, summary_df, window_size_seconds, rq_threshold)

        return summary_df


In [7]:
df_1 = pd.read_csv('visit_data/TR000100_20220203_MiPACE_visit_data.csv') 
test_1 = Test(df_1)
test_1.create_test_summary(window_size_seconds=50, rq_threshold=1.0)   


/var/folders/4v/55nqzrk56yz5gzf4ld7y124h0000gn/T/ipykernel_97452/3605237856.py:547: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0 days 00:44:08' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary_df.loc[stage_df.name, 'Start Time'] = str(start_time).split('.')[0]
/var/folders/4v/55nqzrk56yz5gzf4ld7y124h0000gn/T/ipykernel_97452/3605237856.py:548: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0 days 00:44:58' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary_df.loc[stage_df.name, 'End Time'] = str(end_time).split('.')[0]
/var/folders/4v/55nqzrk56yz5gzf4ld7y124h0000gn/T/ipykernel_97452/3605237856.py:568: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Valu

,Start Time,End Time,Velocity (km/h),Grade (%),Stage (#),RPE (#),Lactate (mmol/L),VO2 (mL/min),VO2 (mL/min/kg),HR (bpm),RQ,True VO2max? T/F,Reach Steady State? T/F,O2 variance
0.0,0 days 00:44:08,0 days 00:44:58,0.0,0.0,0.0,20.0,6.0,370.447520,7.110556,91.722222,0.778889,False,False,15415.731914
1.0,0 days 00:07:11,0 days 00:08:01,6.7,0.0,1.0,10.0,2.8,2410.859211,46.274783,155.391304,0.744783,False,False,23736.624719
2.0,0 days 00:10:11,0 days 00:11:01,7.3,0.0,2.0,11.0,1.7,2538.915670,48.730952,165.238095,0.787619,False,False,11714.800555
3.0,0 days 00:13:09,0 days 00:13:59,8.0,0.0,3.0,13.0,2.7,2817.129149,54.070476,172.714286,0.795238,False,False,22456.772232
4.0,0 days 00:16:10,0 days 00:17:00,8.6,0.0,4.0,15.0,1.6,2913.158795,55.916000,179.920000,0.862800,False,False,10151.186533
5.0,0 days 00:19:10,0 days 00:20:00,9.2,0.0,5.0,16.0,2.3,2974.497736,57.092000,182.266667,0.918333,False,False,53540.316030
6.0,0 days 00:22:11,0 days 00:23:01,9.8,0.0,6.0,18.0,4.1,3070.277362,58.931351,186.054054,0.947297,False,False,39690.289947
7.0,0 days 00:24:03,0 days 00:24:53,9.8,2.0,7.0,18.0,6.8,2966.671479,56.941591,189.295455,0.991364,False,False,6584.325216


In [8]:
#Read in test data into a list of Test objects from 'vist_data' folder
path = r'visit_data'
filenames = glob.glob(os.path.join(path, "*.csv"))
test_list = []
for filename in filenames:
    df = pd.read_csv(filename)
    test = Test(df)
    test_list.append(test)


In [9]:
rows = []
for test in test_list:
    result = test.get_all_metrics('''can add parameters vo2_window_size (default 30)
                                  , vt_window_size (default 30), rq_thresh (default 1.0)''')
    result['name'] = test.name
    rows.append(result)

df = pd.DataFrame(rows).set_index('name')

No deflection point of VE/CO2 found after nadir.


In [451]:
df

,VO2_peak,start_time,end_time,GradePeak,SpeedPeak,MarkerPeak,RQPeak,HRPeak,EEMPeak,Fat%Peak,...,vt1_speed_pet,vt1_grade_pet,vt2_time_pet,vt2_speed_pet,vt2_grade_pet,lt_log_semilog_error,get_VO2_peak_time_metrics_error,vt1_vt2_VE_error,vt1_vt2_vo2_vco2_error,vt1_vt2_pet_error
name,,,,,,,,,,,,,,,,,,,,,
TR000207_20221201,3118.365853,0 days 00:27:40,0 days 00:28:10,4.0,8.8,8.0,0.968519,193.629630,15.391481,2778.148148,...,5.7,0.0,0 days 00:34:53,0.0,0.0,NaN,NaN,NaN,NaN,NaN
TR000199_20220503,4986.884196,0 days 00:20:49,0 days 00:21:19,0.0,9.9,5.0,0.934828,186.689655,24.305862,8153.724138,...,2.0,0.0,0 days 00:22:43,10.5,0.0,NaN,NaN,NaN,NaN,NaN
TR000143_20220804,3632.265363,0 days 00:27:55,0 days 00:28:25,4.0,8.8,8.0,0.993333,187.625000,18.041667,2050.000000,...,6.3,0.0,0 days 00:29:42,8.8,4.0,NaN,NaN,NaN,NaN,NaN
TR000137_20220527,4409.686011,0 days 00:25:55,0 days 00:26:25,2.0,10.5,7.0,0.984783,172.565217,21.610870,1657.826087,...,7.4,0.0,0 days 00:35:34,0.0,0.0,NaN,NaN,NaN,NaN,NaN
TR000194_20220316,4867.916704,0 days 00:28:35,0 days 00:29:05,0.0,11.6,8.0,0.787143,172.142857,22.993214,23175.785714,...,0.0,0.0,0 days 00:28:47,11.6,0.0,zero-size array to reduction operation minimum...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TR000122_20220817,2676.285003,0 days 00:24:16,0 days 00:24:46,0.0,8.8,6.0,0.944000,178.933333,13.194333,3627.166667,...,5.7,0.0,0 days 00:26:42,2.0,0.0,NaN,NaN,NaN,NaN,NaN
TR000223_20220803,3968.881080,0 days 00:29:03,0 days 00:29:33,0.0,11.6,8.0,1.016176,188.970588,19.907941,122.000000,...,7.2,0.0,0 days 00:18:59,9.7,0.0,NaN,NaN,NaN,NaN,NaN
TR000242_20220419,2184.772142,0 days 00:25:25,0 days 00:25:55,0.0,6.4,6.0,0.967500,188.071429,10.715357,1796.678571,...,0.0,0.0,0 days 00:05:09,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 623 entries, TR000207_20221201 to TR000138_20220414
Data columns (total 35 columns):
 #   Column                               Non-Null Count  Dtype          
---  ------                               --------------  -----          
 0   get_VO2_peak_and_time_metrics_error  623 non-null    object         
 1   LT1_abs                              603 non-null    float64        
 2   LT2_abs                              593 non-null    float64        
 3   LT1abs_speed                         603 non-null    float64        
 4   LT2abs_speed                         593 non-null    float64        
 5   LT1_ref                              603 non-null    float64        
 6   LT2_ref                              595 non-null    float64        
 7   LT1ref_speed                         603 non-null    float64        
 8   LT2ref_speed                         595 non-null    float64        
 9   LT1_log                              603 non-null  

In [13]:
error_columns = ['lt_log_semilog_error', 'vt1_vt2_VE_error', 'vt1_vt2_vo2_vco2_error', 'vt1_vt2_pet_error']
rows_with_error = df.dropna(subset=error_columns, how = 'all').index.tolist()


In [14]:
len(rows_with_error)

28